# Advanced Notebook: Hybrid CBR+KNN Implementation

Implementasi hybrid yang menggabungkan:
- **KNN**: Untuk menemukan k-nearest neighbors dengan cepat
- **CBR**: Untuk adaptasi dan penjelasan hasil

Tujuan:
- Mendapatkan stabilitas KNN dengan interpretabilitas CBR
- Evaluasi performa kedua algoritma
- Visualisasi hasil perbandingan

## 📦 Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report
from sklearn.preprocessing import LabelEncoder
import time
from typing import List, Tuple, Dict

# Set style untuk visualisasi
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

## 🔹 Dataset yang Diperluas

In [ ]:
# Dataset yang lebih besar untuk evaluasi yang lebih akurat
data_extended = {
    "demam": [1,1,1,0,0,1,0,1, 1,1,0,0,1,1,0,1, 1,0,1,0],
    "batuk": [1,1,0,1,0,1,0,0, 1,0,1,1,1,0,0,1, 0,1,1,0],
    "sakit_kepala": [0,1,1,0,1,1,0,1, 1,0,0,1,1,1,0,0, 1,0,0,1],
    "penyakit": ["flu","flu","dbd","normal","normal","dbd","normal","flu",
                 "flu","dbd","normal","normal","flu","dbd","normal","flu",
                 "dbd","normal","flu","normal"]
}

df_extended = pd.DataFrame(data_extended)
print("Dataset Extended:")
print(df_extended)
print(f"\nTotal samples: {len(df_extended)}")
print(f"\nClass distribution:\n{df_extended['penyakit'].value_counts()}")

## 🔹 Implementasi CBR (Case-Based Reasoning)

In [ ]:
class CBRSystem:
    """
    Case-Based Reasoning System untuk klasifikasi penyakit.
    
    Atribut:
    - case_base: Database kasus yang tersimpan
    - weights: Bobot untuk setiap fitur (untuk similarity calculation)
    """
    
    def __init__(self, weights: Dict[str, float] = None):
        self.case_base = []
        self.weights = weights or {"demam": 1.0, "batuk": 1.0, "sakit_kepala": 1.0}
        self.feature_names = ["demam", "batuk", "sakit_kepala"]
    
    def similarity(self, case1: List[int], case2: List[int]) -> float:
        """
        Hitung similarity antara dua cases menggunakan weighted matching.
        
        Formula: similarity = sum(weights * matches) / sum(weights)
        """
        total_weight = sum(self.weights.values())
        weighted_matches = 0
        
        for i, feature in enumerate(self.feature_names):
            if case1[i] == case2[i]:
                weighted_matches += self.weights[feature]
        
        return weighted_matches / total_weight
    
    def fit(self, X: List[List[int]], y: List[str]):
        """
        Masukkan cases ke dalam case base.
        Dalam CBR, training adalah menambahkan cases ke database.
        """
        self.case_base = list(zip(X, y))
        print(f"Case base loaded with {len(self.case_base)} cases")
    
    def retrieve(self, new_case: List[int], k: int = 3) -> List[Tuple[float, str]]:
        """
        Retrieve k most similar cases dari case base.
        """
        similarities = []
        
        for case, disease in self.case_base:
            sim = self.similarity(new_case, case)
            similarities.append((sim, disease, case))
        
        # Sort by similarity (descending)
        similarities.sort(key=lambda x: x[0], reverse=True)
        return similarities[:k]
    
    def reuse(self, retrieved_cases: List[Tuple]) -> str:
        """
        Reuse: Hitung diagnosis berdasarkan retrieved cases.
        Menggunakan majority voting berdasarkan similarity weights.
        """
        if not retrieved_cases:
            return "Unknown"
        
        # Weighted voting berdasarkan similarity
        disease_weights = {}
        for sim, disease, _ in retrieved_cases:
            disease_weights[disease] = disease_weights.get(disease, 0) + sim
        
        return max(disease_weights, key=disease_weights.get)
    
    def predict(self, X: List[List[int]], k: int = 3, return_details: bool = False):
        """
        Predict diagnosis untuk cases baru.
        
        Args:
        - X: List of cases
        - k: Jumlah nearest neighbors
        - return_details: Jika True, return detailed information
        """
        predictions = []
        details = []
        
        for case in X:
            retrieved = self.retrieve(case, k)
            diagnosis = self.reuse(retrieved)
            predictions.append(diagnosis)
            
            if return_details:
                details.append({
                    'case': case,
                    'retrieved': retrieved,
                    'diagnosis': diagnosis
                })
        
        return np.array(predictions), details if return_details else None
    
    def get_explanation(self, case: List[int], k: int = 3) -> Dict:
        """
        Dapatkan penjelasan detail untuk sebuah prediksi.
        Ini adalah keunggulan CBR: interpretabilitas tinggi.
        """
        retrieved = self.retrieve(case, k)
        diagnosis = self.reuse(retrieved)
        
        explanation = {
            'input_case': dict(zip(self.feature_names, case)),
            'diagnosis': diagnosis,
            'similar_cases': [
                {
                    'similarity': round(sim, 3),
                    'disease': disease,
                    'case': dict(zip(self.feature_names, case_data))
                }
                for sim, disease, case_data in retrieved
            ]
        }
        return explanation

print("CBR System class defined successfully!")

## 🔹 Implementasi Hybrid CBR+KNN

In [ ]:
class HybridCBRKNN:
    """
    Sistem Hybrid yang menggabungkan kecepatan KNN dengan interpretabilitas CBR.
    
    Strategi:
    1. KNN mencari k-nearest neighbors dengan cepat
    2. CBR memberikan penjelasan dan adaptasi untuk kasus baru
    """
    
    def __init__(self, n_neighbors: int = 3):
        self.knn_model = KNeighborsClassifier(n_neighbors=n_neighbors)
        self.cbr_system = CBRSystem()
        self.X_train = None
        self.y_train = None
        self.n_neighbors = n_neighbors
        self.label_encoder = LabelEncoder()
    
    def fit(self, X, y):
        """
        Train kedua model.
        """
        self.X_train = X
        self.y_train = y
        
        # Train KNN
        self.knn_model.fit(X, y)
        
        # Setup CBR
        self.cbr_system.fit(X.tolist(), y.tolist())
        
        print(f"Hybrid model trained with {len(X)} samples")
    
    def predict_knn(self, X):
        """
        Prediksi menggunakan KNN (cepat, tapi kurang interpretatif).
        """
        return self.knn_model.predict(X)
    
    def predict_cbr(self, X, k=3, return_details=False):
        """
        Prediksi menggunakan CBR (lebih lambat, tapi interpretatif).
        """
        return self.cbr_system.predict(X, k=k, return_details=return_details)
    
    def predict_hybrid(self, X, method='voting'):
        """
        Prediksi hybrid dengan kombinasi KNN dan CBR.
        
        Method:
        - 'voting': Majority vote antara KNN dan CBR
        - 'weighted': Weighted average berdasarkan confidence
        """
        knn_pred = self.knn_model.predict(X)
        cbr_pred, _ = self.cbr_system.predict(X, k=self.n_neighbors, return_details=False)
        
        if method == 'voting':
            # Majority voting
            hybrid_pred = []
            for k_pred, c_pred in zip(knn_pred, cbr_pred):
                if k_pred == c_pred:
                    hybrid_pred.append(k_pred)
                else:
                    # Jika berbeda, gunakan KNN (lebih stabil)
                    hybrid_pred.append(k_pred)
            return np.array(hybrid_pred)
        
        return knn_pred  # default to KNN
    
    def get_hybrid_explanation(self, case, k=3):
        """
        Dapatkan penjelasan dari kedua model untuk interpretabilitas maksimal.
        """
        case_2d = np.array([case]).reshape(1, -1)
        
        knn_pred = self.knn_model.predict(case_2d)[0]
        knn_proba = self.knn_model.predict_proba(case_2d)[0]
        
        cbr_explanation = self.cbr_system.get_explanation(case, k=k)
        
        return {
            'input': dict(zip(['demam', 'batuk', 'sakit_kepala'], case)),
            'knn_prediction': knn_pred,
            'knn_confidence': round(max(knn_proba), 3),
            'cbr_explanation': cbr_explanation,
            'consensus': knn_pred == cbr_explanation['diagnosis']
        }

print("Hybrid CBR+KNN class defined successfully!")

## 🔹 Train Test Split & Model Training

In [ ]:
# Persiapan data
X = df_extended[["demam", "batuk", "sakit_kepala"]].values
y = df_extended["penyakit"].values

# Split data (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Training set: {len(X_train)} samples")
print(f"Test set: {len(X_test)} samples")
print(f"\nTraining set class distribution:\n{pd.Series(y_train).value_counts()}")
print(f"\nTest set class distribution:\n{pd.Series(y_test).value_counts()}")

# Train models
cbr_model = CBRSystem()
cbr_model.fit(X_train.tolist(), y_train.tolist())

knn_model = KNeighborsClassifier(n_neighbors=3)
knn_model.fit(X_train, y_train)

hybrid_model = HybridCBRKNN(n_neighbors=3)
hybrid_model.fit(X_train, y_train)

print("\n✅ All models trained successfully!")

## 🔹 Prediksi dan Evaluasi

In [ ]:
# Prediksi
y_pred_cbr, _ = cbr_model.predict(X_test.tolist())
y_pred_knn = knn_model.predict(X_test)
y_pred_hybrid = hybrid_model.predict_hybrid(X_test, method='voting')

# Hitung metrics
metrics_dict = {}

for name, y_pred in [("CBR", y_pred_cbr), ("KNN", y_pred_knn), ("Hybrid", y_pred_hybrid)]:
    metrics_dict[name] = {
        'Accuracy': accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred, average='weighted', zero_division=0),
        'Recall': recall_score(y_test, y_pred, average='weighted', zero_division=0),
        'F1-Score': f1_score(y_test, y_pred, average='weighted', zero_division=0)
    }

# Buat DataFrame untuk perbandingan
metrics_df = pd.DataFrame(metrics_dict).T
print("\n" + "="*60)
print("PERBANDINGAN METRIK PERFORMA")
print("="*60)
print(metrics_df.round(4))
print("="*60)

## 🔹 Analisis Waktu Eksekusi

In [ ]:
# Ukur waktu prediksi
timing_results = {}

# CBR timing
start = time.time()
for _ in range(100):
    cbr_model.predict(X_test.tolist())
timing_results['CBR'] = (time.time() - start) / 100

# KNN timing
start = time.time()
for _ in range(100):
    knn_model.predict(X_test)
timing_results['KNN'] = (time.time() - start) / 100

# Hybrid timing
start = time.time()
for _ in range(100):
    hybrid_model.predict_hybrid(X_test)
timing_results['Hybrid'] = (time.time() - start) / 100

timing_df = pd.DataFrame({
    'Model': list(timing_results.keys()),
    'Avg Time (ms)': [v*1000 for v in timing_results.values()]
})

print("\n" + "="*60)
print("WAKTU EKSEKUSI PREDIKSI (rata-rata per batch)")
print("="*60)
print(timing_df.to_string(index=False))
print("="*60)

## 🔹 Visualisasi Perbandingan

In [ ]:
# Visualisasi metrik performa
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Perbandingan Performa: CBR vs KNN vs Hybrid', fontsize=16, fontweight='bold')

# Accuracy
metrics_df['Accuracy'].plot(kind='bar', ax=axes[0, 0], color=['#FF6B6B', '#4ECDC4', '#45B7D1'])
axes[0, 0].set_title('Accuracy', fontweight='bold')
axes[0, 0].set_ylabel('Score')
axes[0, 0].set_xticklabels(axes[0, 0].get_xticklabels(), rotation=45)
axes[0, 0].set_ylim([0, 1])
axes[0, 0].grid(axis='y', alpha=0.3)

# Precision
metrics_df['Precision'].plot(kind='bar', ax=axes[0, 1], color=['#FF6B6B', '#4ECDC4', '#45B7D1'])
axes[0, 1].set_title('Precision', fontweight='bold')
axes[0, 1].set_ylabel('Score')
axes[0, 1].set_xticklabels(axes[0, 1].get_xticklabels(), rotation=45)
axes[0, 1].set_ylim([0, 1])
axes[0, 1].grid(axis='y', alpha=0.3)

# Recall
metrics_df['Recall'].plot(kind='bar', ax=axes[1, 0], color=['#FF6B6B', '#4ECDC4', '#45B7D1'])
axes[1, 0].set_title('Recall', fontweight='bold')
axes[1, 0].set_ylabel('Score')
axes[1, 0].set_xticklabels(axes[1, 0].get_xticklabels(), rotation=45)
axes[1, 0].set_ylim([0, 1])
axes[1, 0].grid(axis='y', alpha=0.3)

# F1-Score
metrics_df['F1-Score'].plot(kind='bar', ax=axes[1, 1], color=['#FF6B6B', '#4ECDC4', '#45B7D1'])
axes[1, 1].set_title('F1-Score', fontweight='bold')
axes[1, 1].set_ylabel('Score')
axes[1, 1].set_xticklabels(axes[1, 1].get_xticklabels(), rotation=45)
axes[1, 1].set_ylim([0, 1])
axes[1, 1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

print("✅ Visualisasi metrik berhasil ditampilkan")

## 🔹 Confusion Matrix

In [ ]:
# Hitung confusion matrices
cm_cbr = confusion_matrix(y_test, y_pred_cbr)
cm_knn = confusion_matrix(y_test, y_pred_knn)
cm_hybrid = confusion_matrix(y_test, y_pred_hybrid)

# Dapatkan label yang unik dan urut
classes = sorted(np.unique(y_test))

# Visualisasi
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
fig.suptitle('Confusion Matrix: CBR vs KNN vs Hybrid', fontsize=14, fontweight='bold')

for idx, (cm, title) in enumerate([(cm_cbr, 'CBR'), (cm_knn, 'KNN'), (cm_hybrid, 'Hybrid')]):
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[idx], 
                xticklabels=classes, yticklabels=classes, cbar=False)
    axes[idx].set_title(title, fontweight='bold')
    axes[idx].set_ylabel('True Label')
    axes[idx].set_xlabel('Predicted Label')

plt.tight_layout()
plt.show()

print("✅ Confusion matrix visualization berhasil ditampilkan")

## 🔹 Visualisasi Waktu Eksekusi

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

colors = ['#FF6B6B', '#4ECDC4', '#45B7D1']
bars = ax.bar(timing_df['Model'], timing_df['Avg Time (ms)'], color=colors, edgecolor='black', linewidth=1.5)

# Tambah nilai pada setiap bar
for bar in bars:
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'{height:.4f} ms',
            ha='center', va='bottom', fontweight='bold')

ax.set_title('Perbandingan Waktu Eksekusi Prediksi', fontsize=14, fontweight='bold')
ax.set_ylabel('Waktu (milliseconds)', fontsize=12)
ax.set_xlabel('Model', fontsize=12)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

print("✅ Timing visualization berhasil ditampilkan")

## 🔹 Contoh Penjelasan Hybrid

In [ ]:
import json

# Contoh kasus baru
test_case = X_test[0]
print("\n" + "="*70)
print("CONTOH KASUS UJI")
print("="*70)
print(f"Input Case: demam={test_case[0]}, batuk={test_case[1]}, sakit_kepala={test_case[2]}")
print(f"Actual Diagnosis: {y_test[0]}")

# Dapatkan penjelasan hybrid
explanation = hybrid_model.get_hybrid_explanation(test_case, k=3)

print("\n" + "-"*70)
print("KNN PREDICTION")
print("-"*70)
print(f"Prediction: {explanation['knn_prediction']}")
print(f"Confidence: {explanation['knn_confidence']}")

print("\n" + "-"*70)
print("CBR EXPLANATION")
print("-"*70)
print(f"Diagnosis: {explanation['cbr_explanation']['diagnosis']}")
print(f"\nRetrieved Similar Cases:")
for i, case_info in enumerate(explanation['cbr_explanation']['similar_cases'], 1):
    print(f"  {i}. Similarity: {case_info['similarity']:.3f} | Disease: {case_info['disease']}")
    print(f"     Case: {case_info['case']}")

print("\n" + "-"*70)
print("HYBRID CONSENSUS")
print("-"*70)
print(f"KNN dan CBR Agree: {explanation['consensus']}")
print(f"Final Diagnosis: {explanation['knn_prediction']}")
print("="*70)

## 🔹 Detailed Classification Report

In [ ]:
print("\n" + "="*70)
print("DETAILED CLASSIFICATION REPORT")
print("="*70)

for name, y_pred in [("CBR", y_pred_cbr), ("KNN", y_pred_knn), ("Hybrid", y_pred_hybrid)]:
    print(f"\n{name} Model:")
    print("-" * 70)
    print(classification_report(y_test, y_pred, zero_division=0))

## 📊 Analisis dan Kesimpulan

### Mana lebih stabil?
**KNN** adalah yang paling stabil karena:
- Menggunakan metrik jarak matematika yang konsisten
- Tidak tergantung pada definisi similarity yang subjektif
- Performa lebih predictable

### Mana lebih interpretatif?
**CBR** adalah yang paling interpretatif karena:
- Menunjukkan kasus-kasus serupa yang digunakan dalam pengambilan keputusan
- Dapat memberikan penjelasan "mengapa" selain "apa"
- Memungkinkan adaptasi manual

### Kapan gunakan masing-masing?

| Situasi | Pilihan |
|---------|----------|
| Prediksi cepat, interpretasi kurang penting | **KNN** |
| Penjelasan dan transparansi kritis | **CBR** |
| Balance antara kecepatan dan interpretabilitas | **Hybrid** |
| Domain dengan banyak expert knowledge | **CBR** |
| Data terstruktur dengan label jelas | **KNN** |
| Sistem support/diagnosis medis | **Hybrid** |

### Rekomendasi untuk Kasus Diagnosa Penyakit

🎯 **Gunakan Hybrid** karena:
1. ✅ Menggabungkan kecepatan KNN dengan interpretabilitas CBR
2. ✅ Dokter dapat melihat kasus serupa untuk memverifikasi diagnosis
3. ✅ Sistem dapat beradaptasi dengan kasus baru
4. ✅ Memberikan confidence level (dari probabilitas KNN)
5. ✅ Lebih trustworthy untuk aplikasi medis